# WatchHub — Offline Embedding Generation

Generates the two artifacts that ship with the repository:

- `tmdb_movies_clean.csv` — the filtered movie catalogue, seeded into MongoDB at startup
- `overview_vectors.npz` — sentence embeddings of every plot overview, loaded into memory at startup

Run once on a Kaggle GPU. The model is never present in the application container —
the backend only does dot products against these precomputed vectors.

**Source dataset:** [TMDB Movies Dataset](https://www.kaggle.com/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies) by asaniczka.

## 1. Inspect the raw data

Checking field formats and coverage before filtering. The language counts confirm the
catalogue is genuinely multilingual, and the keyword coverage (~19% missing) is why
tag similarity weights genres higher than keywords.

In [1]:
import pandas as pd

FILENAME = "/kaggle/input/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies/TMDB_movie_dataset_v11.csv"
df = pd.read_csv(FILENAME)
print("loaded:", df.shape)

good = df[(df["vote_count"] >= 25) & (df["genres"].notna()) & (df["overview"].notna())].copy()

print("\nGENRES sample:", good["genres"].head(3).tolist())
print("KEYWORDS sample:", good["keywords"].head(3).tolist())
print("KEYWORDS empty:", round(100*good["keywords"].isna().mean(),1), "%")

print("\nruntime dtype:", good["runtime"].dtype, "| sample:", good["runtime"].head(3).tolist())
print("release_date sample:", good["release_date"].head(3).tolist())
print("release_date empty:", good["release_date"].isna().sum())

print("\nlanguages with >=20 films:")
vc = good["original_language"].value_counts()
print(vc[vc>=20])

loaded: (1452309, 24)

GENRES sample: ['Action, Science Fiction, Adventure', 'Adventure, Drama, Science Fiction', 'Drama, Action, Crime, Thriller']
KEYWORDS sample: ['rescue, mission, dream, airplane, paris, france, virtual reality, kidnapping, philosophy, spy, allegory, manipulation, car crash, heist, memory, architecture, los angeles, california, dream world, subconscious', 'rescue, future, spacecraft, race against time, artificial intelligence (a.i.), nasa, time warp, dystopia, expedition, space travel, wormhole, famine, black hole, quantum mechanics, family relationships, space, robot, astronaut, scientist, single father, farmer, space station, curious, space adventure, time paradox, thoughtful, time-manipulation, father daughter relationship, 2060s, cornfield, time manipulation, complicated', 'joker, sadism, chaos, secret identity, crime fighter, superhero, anti hero, scarecrow, based on comic, vigilante, organized crime, tragic hero, anti villain, criminal mastermind, district at

## 2. Confirm GPU

Encoding 43k overviews on CPU takes several minutes. On a T4 it's under a minute.

In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

GPU available: True
Device: Tesla T4


In [3]:
!pip install sentence-transformers -q

## 3. Filter and export the catalogue

Four filters:

- `vote_count >= 25` — below this the rating isn't meaningful
- non-empty `genres` and `overview` — both are required for matching, a row without them can't be scored
- `adult == False`

`poster_path` is kept so the frontend can build poster URLs at display time.
The same `good` dataframe feeds the embedding step below, so the CSV and the
vectors always describe the same set of movies.

In [4]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

FILENAME = "/kaggle/input/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies/TMDB_movie_dataset_v11.csv"
df = pd.read_csv(FILENAME)
print("full dataset:", df.shape)

keep = ["id", "title", "original_language", "release_date", "runtime",
        "vote_average", "vote_count", "popularity", "genres", "keywords",
        "overview", "poster_path"]

good = df[(df["vote_count"] >= 25) &
          (df["genres"].notna()) &
          (df["overview"].notna()) &
          (df["adult"] == False)][keep].copy()

print("after filters:", good.shape)
good.to_csv("tmdb_movies_clean.csv", index=False)
print("saved tmdb_movies_clean.csv")

full dataset: (1453952, 24)
after filters: (43081, 12)
saved tmdb_movies_clean.csv


## 4. Generate and save the embeddings

`normalize_embeddings=True` scales every vector to unit length. That means cosine
similarity reduces to a plain dot product at query time — no division needed, which
is why `vectors.py` has a one-line `cosine()`.

Saved as float16 to halve the file size. The backend upcasts to float32 on load,
since that's faster on CPU.

In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("model loaded")

overviews = good["overview"].tolist()
print(f"encoding {len(overviews)} overviews...")

vecs = model.encode(
    overviews,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print("vectors shape:", vecs.shape)

np.savez_compressed(
    "overview_vectors.npz",
    ids=good["id"].to_numpy(),
    vectors=vecs.astype(np.float16),
)
print("saved overview_vectors.npz")

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model loaded
encoding 43081 overviews...


vectors shape: (43081, 384)
saved overview_vectors.npz
